# Collaboration step 1, notebook 3: the scorer on student texts, then the six features

The scorer trained in notebook 2 is applied to the texts students wrote. **There are no human labels for these texts**, so this is a sanity check of behaviour, not a validation. The second part builds the six
features of the proposal for every student-sprint and compares heads on the four-criterion Collaboration ratings. Those ratings were written by the assistant (see `dataset/reference_ratings/README.md`),
and the simulated students were written by the assistant too, so the second part is a **pipeline test**, not evidence of accuracy.

In [1]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_columns", 30)
import data, substance
from cleaning import substance as rule_score, clean_text, is_padding, APPROVAL_ONLY
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
od, rh = data.public_pool()
od = od.reset_index(drop=True); rh = rh.reset_index(drop=True)
CS = substance.combos(od.group.to_numpy())
OUTER = {}
for o, i, tr, va, te in CS:
    OUTER[o] = (np.setdiff1d(np.arange(len(od)), te), te)
def guarded(original):
    c = clean_text(original)
    return len(c.split()) < 3 or bool(APPROVAL_ONLY.match(c)) or is_padding(c)

import joblib
bundle = joblib.load(RESULTS / "substance_scorer.joblib")
def score_texts(originals):
    clean = [clean_text(t) for t in originals]
    E = substance.embed(bundle["encoder"], clean)
    if bundle["variant"] == "emb+meta":
        M = substance.meta(list(originals), clean); E = np.hstack([E, (M - bundle["meta_mean"]) / (bundle["meta_std"] + 1e-9)])
    p = bundle["head"].predict_proba(E)[:, 1]
    return np.where([guarded(t) for t in originals], 0.0, p)

## 1. Student texts

In [2]:
st_ = data.student_texts()
st_["score"] = score_texts(st_.text.tolist()); st_["rule score"] = [rule_score(t) for t in st_.text]; st_["words"] = st_.clean.str.split().str.len()
display(st_.groupby(["source", "kind"]).agg(texts=("score", "size"), mean_score=("score", "mean"), zero_share=("score", lambda s: (s == 0).mean())).round(3))
display(st_[st_.source == "real"][["kind", "text", "words", "score"]].round(3))

texts  mean_score  zero_share
source    kind                                  
real      comment      1       0.707       0.000
          review      12       0.000       1.000
simulated comment     35       0.492       0.029
          review     105       0.537       0.257

,kind,text,words,score
0,review,All Good.,2,0.000
1,review,All good..,2,0.000
2,review,good.,1,0.000
3,review,well done.,2,0.000
4,review,ok,1,0.000
5,review,ok,1,0.000
6,review,ok,1,0.000
7,review,good,1,0.000
8,review,ok,1,0.000
9,review,ok,1,0.000


In [3]:
sim = st_[st_.source == "simulated"]
print("simulated texts:", len(sim), "| Spearman of the model score with the rule score:", round(float(spearmanr(sim.score, sim["rule score"])[0]), 2))
display(sim.groupby(pd.cut(sim.words, [-1, 2, 6, 15, 30, 400]))["score"].agg(["count", "mean"]).round(3))
display(sim.sort_values("score")[["text", "words", "score"]].round(3).iloc[[0, 1, 2, 3, -4, -3, -2, -1]])

simulated texts: 140 | Spearman of the model score with the rule score: 0.84


,count,mean
words,,
"(-1, 2]",28,0.000
"(2, 6]",17,0.508
"(6, 15]",35,0.574
"(15, 30]",36,0.712
"(30, 400]",24,0.801


,text,words,score
19,ok,1,0.000
17,looks good,2,0.000
25,nice,1,0.000
23,ok,1,0.000
71,Complete() starts a transaction but the idempotency check for a repeated req...,36,0.895
15,The admin list reloads every time the form changes because the effect depend...,33,0.903
84,"Cookie flags are right now, thanks. The auth test for a wrong password is go...",21,0.909
83,The login route stores the session id in a cookie but I do not see httpOnly ...,36,0.915


## 2. The six features per student-sprint

`Co_PRC_depth` is now the mean model score of the student's reviews on other people's pull requests (0 when there are none); `Co_IC_depth` the same for discussion comments; `Co_MC`, `Co_PCount`, `Co_CCount`, `Co_IR`
as defined in `cleaning.py` and stored in the clean dataset. The label is the mean of the four Collaboration criteria of the ratings, rescaled to 0-1.

In [4]:
FM = data.FM
final = pd.read_csv(FM / "dataset" / "clean" / "final_dataset_clean.csv", low_memory=False)
ref = pd.read_csv(FM / "dataset" / "reference_ratings" / "reference_ratings.csv"); hum = pd.read_csv(FM / "dataset" / "ground_truth" / "human_ratings.csv")
CO = ["Co_review_engagement", "Co_discussion_substance", "Co_team_interaction", "Co_follow_through"]
rate = pd.concat([ref, hum]).drop_duplicates("record_id").set_index("record_id")
final = final[final.record_id.isin(rate.index)].copy()
final["y"] = ((rate.loc[final.record_id, CO].astype(float).mean(axis=1) - 1) / 4).to_numpy()
# model-based depth per (student, sprint)
rv = pd.concat([pd.read_csv(data.SIM / "reviews_simulated.csv").assign(src="simulated"), pd.read_csv(data.EVID / "pr_reviews.csv").assign(src="real")]).fillna("")
rv = rv[rv.body.str.strip() != ""]
rv["score"] = score_texts(rv.body.tolist())
depth = rv.groupby(["sprint_id", "reviewer_id"]).score.mean()
final["PRC_model"] = [float(depth.get((s, u), 0.0)) for s, u in zip(final.sprint_id, final.student_id)]
final["PRC_rule"] = final.Co_PRC_depth
feat_model = ["PRC_model", "Co_IC_depth", "Co_MC", "Co_PCount", "Co_CCount", "Co_IR"]
print(final.groupby("source").agg(records=("y", "size"), with_reviews=("PRC_model", lambda s: int((s > 0).sum())), label_mean=("y", "mean"), label_sd=("y", "std")).round(3))

           records  with_reviews  label_mean  label_sd
source                                                
real           144             0       0.217     0.186
simulated      175            30       0.336     0.201


## 3. Heads on the six features (grouped by student)

Ridge, Random Forest and Gradient Boosting against two baselines: the review count alone and the mean. Grouped five-fold CV by student. Reported separately for the real students (almost nothing varies) and
the simulated ones (a pipeline test).

In [5]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
def cv_mae(d, cols, make):
    p = np.zeros(len(d))
    for tr, te in GroupKFold(5).split(d, groups=d.student_id):
        m = make(); m.fit(d[cols].iloc[tr], d.y.iloc[tr]); p[te] = np.clip(m.predict(d[cols].iloc[te]), 0, 1)
    return float(np.abs(p - d.y).mean()), float(spearmanr(p, d.y)[0]) if p.std() > 0 else float("nan")
from sklearn.impute import SimpleImputer
imp = lambda: SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)   # the response rate is undefined for students with no own pull request
HEADS = {"mean": lambda: make_pipeline(imp(), StandardScaler(), Ridge(alpha=1e9)), "Ridge (6 features)": lambda: make_pipeline(imp(), StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 11))),
         "Random Forest": lambda: make_pipeline(imp(), RandomForestRegressor(200, min_samples_leaf=3, random_state=0, n_jobs=1)),
         "Gradient Boosting": lambda: make_pipeline(imp(), GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=0))}
rows = []
for src in ("real", "simulated"):
    d = final[final.source == src].reset_index(drop=True)
    if len(d) < 20:
        continue
    for name, mk in HEADS.items():
        mae, rho = cv_mae(d, feat_model, mk); rows.append({"students": src, "head": name, "features": "model depth + 5", "MAE": mae, "Spearman": rho})
    for label, cols in (("review count only (Co_PCount)", ["Co_PCount"]),):
        mae, rho = cv_mae(d, cols, HEADS["Ridge (6 features)"]); rows.append({"students": src, "head": "Ridge", "features": label, "MAE": mae, "Spearman": rho})
    mae, rho = cv_mae(d.assign(PRC_model=d.PRC_rule), ["PRC_model"] + feat_model[1:], HEADS["Ridge (6 features)"]); rows.append({"students": src, "head": "Ridge", "features": "rule depth + 5 (old)", "MAE": mae, "Spearman": rho})
pd.DataFrame(rows).round(3)

,students,head,features,MAE,Spearman
0,real,mean,model depth + 5,0.160,-0.044
1,real,Ridge (6 features),model depth + 5,0.111,0.502
2,real,Random Forest,model depth + 5,0.107,0.482
3,real,Gradient Boosting,model depth + 5,0.106,0.512
4,real,Ridge,review count only (Co_PCount),0.144,0.132
5,real,Ridge,rule depth + 5 (old),0.111,0.502
6,simulated,mean,model depth + 5,0.131,-0.039
7,simulated,Ridge (6 features),model depth + 5,0.044,0.739
8,simulated,Random Forest,model depth + 5,0.042,0.705
9,simulated,Gradient Boosting,model depth + 5,0.036,0.731


## 4. Reading the results

**Student texts.** Twelve of the 13 real texts are approvals ("ok", "good", "All good.") and the guard layer scores all of them 0; the one real comment ("resolve the merge conflicts in this pull request") scores 0.71. The simulated reviews (written by the assistant) behave sensibly: texts of up to 2 words score 0, and the score rises with length and technical content to about 0.9 for the long, specific reviews. The trained scorer's score correlates 0.84 with the rule score on the simulated texts, which fits notebook 2: the trained scorer adds little beyond the rules. None of this is validation, because there are no human labels for student comments.

**Learned combiner, as a pipeline test.**
- On the simulated students (150 records with signal, grouped by student), Ridge, Random Forest and Gradient Boosting on the six measures reach a mean absolute error of 0.04, against 0.13 for the mean and 0.08 for review count alone. But the old rule-based depth feature does equally well (0.044) as the trained scorer's depth, so the scorer contributes nothing. Both the simulated reviews and the ratings were written by the assistant, so this only shows that the pipeline runs and that the learned combiner can recover the rules the data were built from.
- On the real students no record has a review that scores above 0 (every real review is an approval), so the six measures carry almost no information. The learned heads still reach a Spearman of about 0.5, but it comes from the other measures (team pull-request use, merges), and the ratings used as labels were written by the assistant from the same signals. This is **not** evidence that Collaboration works.

**Consequence for the design.** This supports the protocol's decision to specify the combination (fixed, documented weights) instead of fitting it, with the evidence gate for teams that show no review activity, and to state plainly that Collaboration is not validated on students.